# Evaluación final con FOM-5 v2 autoritativo

Ejecutar en Colab limpio después de guardar las 31 generaciones. Qwen3-14B solo
actúa como juez post-hoc. Se conservan sin cambios prompt, parser, retries,
topes y agregación del notebook original. Se calibra con sus cinco casos.
El CSV congelado conserva sus resultados; las 31 respuestas nuevas se puntúan aquí.
No modificar el juez después de observar estos resultados.


In [ ]:
%pip -q install "transformers==4.51.3" "accelerate==1.6.0" "bitsandbytes==0.45.5"


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import json, re, time, gc, hashlib, importlib.metadata as metadata
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from huggingface_hub import HfApi
assert torch.cuda.is_available()
PACKAGE=Path('/content/drive/MyDrive/IA/entrega2_qlora')
PROJECT=Path('/content/drive/MyDrive/IA/qwen25_qlora_or_v2')
FINAL_RUN=PROJECT/'final/user_only_run01'
OUT_DIR=FINAL_RUN/'evaluation'
OUT_DIR.mkdir(exist_ok=True)
old_df=pd.read_csv(PACKAGE/'codex_missing_assets/resultados_fom5_local.csv')
new_df=pd.read_json(FINAL_RUN/'benchmark/final_generations_31.jsonl',lines=True)
assert len(new_df)==31 and new_df['id'].nunique()==31
assert set(new_df['variant'])=={'qlora_final'}
GOLD=[json.loads(s) for s in (PACKAGE/'codex_missing_assets/benchmark_30_v2.jsonl').read_text(encoding='utf-8').splitlines()]
GOLD.append(json.loads((PACKAGE/'results/baseline/caso_entrega1_evaluation_only.json').read_text(encoding='utf-8')))
GOLD_LOOKUP={r['id']:r for r in GOLD}
generation_only=[json.loads(s) for s in (PACKAGE/'codex_missing_assets/benchmark_30_generation_only.jsonl').read_text(encoding='utf-8').splitlines()]
for row in generation_only:
    assert row['problem']==GOLD_LOOKUP[row['id']]['problem']
for row in new_df.to_dict('records'):
    assert row['problem_sha256']==hashlib.sha256(GOLD_LOOKUP[row['id']]['problem'].encode()).hexdigest()
assert set(new_df['id'])==set(GOLD_LOOKUP)
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
fingerprint={'generations_sha256':sha(FINAL_RUN/'benchmark/final_generations_31.jsonl'),
    'evaluator_sha256':sha(PACKAGE/'codex_missing_assets/evaluador_fom5_local.ipynb'),
    'gold_sha256':sha(PACKAGE/'codex_missing_assets/benchmark_30_v2.jsonl'),
    'diagnostic_gold_sha256':sha(PACKAGE/'results/baseline/caso_entrega1_evaluation_only.json'),
    'judge_revision':HfApi().model_info('Qwen/Qwen3-14B').sha,
    'gpu':torch.cuda.get_device_name(0),
    'versions':{p:metadata.version(p) for p in ['torch','transformers','accelerate','bitsandbytes']}}
fp=OUT_DIR/'evaluation_manifest.json'
if fp.exists(): assert json.loads(fp.read_text())==fingerprint, 'Estado distinto: no mezclar evaluaciones.'
else: fp.write_text(json.dumps(fingerprint,indent=2),encoding='utf-8')


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

JUDGE_MODEL = "Qwen/Qwen3-14B"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(
    JUDGE_MODEL,
    use_fast=True,
    revision=fingerprint["judge_revision"],
)

model = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL,
    quantization_config=quant_config,
    revision=fingerprint["judge_revision"],
    device_map={"": 0},
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

model.eval()

allocated = torch.cuda.memory_allocated() / 1024**3
reserved = torch.cuda.memory_reserved() / 1024**3

print("Judge:", JUDGE_MODEL)
print(f"VRAM asignada: {allocated:.2f} GB")
print(f"VRAM reservada: {reserved:.2f} GB")


In [ ]:
SYSTEM_JUDGE = r"""
Eres un juez estricto de formulaciones de Investigación de Operaciones.

TAREA DEL CANDIDATO:
Traducir un problema PL/MILP desde lenguaje natural a una formulación matemática.
Debe formular, NO resolver numéricamente.

REGLAS:
1. Evalúa solo lo escrito. No completes ecuaciones faltantes.
2. Acepta formulaciones matemáticamente equivalentes al gold aunque cambien símbolos o técnica.
3. Una restricción esencial omitida, invertida o aplicada a la variable equivocada es missing/wrong.
4. Una explicación, algoritmo, código o solución numérica no sustituye una formulación matemática.
5. Si falta una variable decisional esencial, decision_domains <= 2.
6. Si el objetivo tiene el sentido correcto pero omite un término esencial, objective <= 2.
7. Si no hay objetivo matemático, objective <= 1.
8. Si no existe formulación matemática identificable, has_formulation=false.
9. Contradicciones entre ecuaciones penalizan constraints_logic y algebra_validity.
10. Si formula correctamente pero además resuelve la instancia, generalization <= 4.
11. Si sustituye la formulación por una solución numérica, has_formulation=false.
12. No premies prosa ni castigues notación distinta si es coherente.

ESCALA 0-5:
0 ausente/totalmente inválido
1 falla fundamental
2 estructura reconocible con error esencial
3 mayormente correcta, pero falta/falla una pieza material
4 correcta salvo detalle menor
5 completa y matemáticamente correcta/equivalente

Estados de requisito permitidos:
met, partial, missing, wrong

Devuelve SOLO JSON válido. Sin markdown y sin explicaciones largas.
"""

DIMENSIONS = [
    "decision_domains",
    "objective",
    "constraints_logic",
    "algebra_validity",
    "generalization",
]

def judge_prompt(item, candidate):
    req_lines = "\n".join(
        f'{r["id"]} [{r["kind"]}]: {r["text"]}'
        for r in item["requirements"]
    )

    req_template = ", ".join(
        f'"{r["id"]}":"met|partial|missing|wrong"'
        for r in item["requirements"]
    )

    return f"""
ENUNCIADO:
{item["problem"]}

REQUISITOS:
{req_lines}

GOLD DE REFERENCIA:
{item["reference_model"]}

RESPUESTA CANDIDATA:
{candidate}

Responde exactamente con esta estructura:
{{
  "has_formulation": true,
  "requirements": {{{req_template}}},
  "scores": {{
    "decision_domains": 0,
    "objective": 0,
    "constraints_logic": 0,
    "algebra_validity": 0,
    "generalization": 0
  }},
  "fatal_errors": ["error breve si corresponde"]
}}
"""


In [ ]:
MAX_INPUT_TOKENS = 12000
MAX_NEW_TOKENS = 600

def _chat_text(messages):
    # Qwen3 permite desactivar thinking explícito.
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

def extract_json(text):
    text = str(text).strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)

    if "</think>" in text:
        text = text.split("</think>", 1)[1].strip()

    start = text.find("{")
    if start < 0:
        raise ValueError("No se encontró inicio de JSON.")

    depth = 0
    in_string = False
    escape = False

    for i in range(start, len(text)):
        ch = text[i]

        if in_string:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
            continue

        if ch == '"':
            in_string = True
        elif ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return json.loads(text[start:i+1])

    raise ValueError("JSON incompleto.")

def validate_parsed(item, parsed):
    if not isinstance(parsed, dict):
        raise ValueError("La respuesta parseada no es un objeto.")

    if "has_formulation" not in parsed:
        raise ValueError("Falta has_formulation.")

    reqs = parsed.get("requirements")
    if not isinstance(reqs, dict):
        raise ValueError("requirements debe ser un objeto {R1: estado, ...}.")

    expected = {r["id"] for r in item["requirements"]}
    got = set(reqs)

    if expected != got:
        raise ValueError(
            f"IDs de requisitos incorrectos. Esperados={expected}; recibidos={got}"
        )

    allowed = {"met", "partial", "missing", "wrong"}
    bad_status = {
        rid: status
        for rid, status in reqs.items()
        if str(status).lower().strip() not in allowed
    }
    if bad_status:
        raise ValueError(f"Estados inválidos: {bad_status}")

    scores = parsed.get("scores")
    if not isinstance(scores, dict):
        raise ValueError("Falta scores.")

    for dim in DIMENSIONS:
        if dim not in scores:
            raise ValueError(f"Falta score: {dim}")
        value = float(scores[dim])
        if not (0 <= value <= 5):
            raise ValueError(f"Score fuera de rango en {dim}: {value}")

    if "fatal_errors" not in parsed:
        parsed["fatal_errors"] = []

    return parsed

@torch.inference_mode()
def judge_local(item, candidate, max_retries=2):
    prompt = judge_prompt(item, candidate)
    messages = [
        {"role": "system", "content": SYSTEM_JUDGE},
        {"role": "user", "content": prompt},
    ]

    last_error = None
    last_raw = None

    for attempt in range(max_retries):
        local_messages = list(messages)
        if attempt > 0:
            local_messages.append({
                "role": "user",
                "content": (
                    "La salida anterior no pudo parsearse. "
                    "Repite el mismo juicio y devuelve SOLO el JSON solicitado, "
                    "completo y sin texto adicional."
                ),
            })

        text = _chat_text(local_messages)
        inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False)

        n_input = int(inputs["input_ids"].shape[1])
        if n_input > MAX_INPUT_TOKENS:
            raise ValueError(
                f"Prompt de {n_input} tokens supera MAX_INPUT_TOKENS={MAX_INPUT_TOKENS}. "
                "No se truncó silenciosamente."
            )

        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        try:
            output = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                use_cache=True,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

            generated = output[0, inputs["input_ids"].shape[1]:]
            raw = tokenizer.decode(generated, skip_special_tokens=True).strip()
            last_raw = raw

            parsed = validate_parsed(item, extract_json(raw))
            return parsed, raw, n_input, int(generated.shape[0])

        except Exception as e:
            last_error = str(e)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    raise RuntimeError(
        f"Judge local falló tras {max_retries} intentos: {last_error}\n"
        f"Última salida: {last_raw}"
    )


In [ ]:
STATUS_VALUE = {
    "met": 1.0,
    "partial": 0.5,
    "missing": 0.0,
    "wrong": 0.0,
}

def safe_score(value):
    return max(0.0, min(5.0, float(value)))

def aggregate_v2(item, parsed):
    status = {
        rid: STATUS_VALUE[str(st).lower().strip()]
        for rid, st in parsed["requirements"].items()
    }

    by_kind = {"objective": [], "constraint": [], "domain": []}

    for req in item["requirements"]:
        by_kind[req["kind"]].append(
            status.get(req["id"], 0.0)
        )

    obj_cov = np.mean(by_kind["objective"]) if by_kind["objective"] else 1.0
    con_cov = np.mean(by_kind["constraint"]) if by_kind["constraint"] else 1.0
    dom_cov = np.mean(by_kind["domain"]) if by_kind["domain"] else 1.0

    raw = parsed["scores"]

    decision = safe_score(raw["decision_domains"])
    objective = min(safe_score(raw["objective"]), 5.0 * obj_cov)
    constraints = min(safe_score(raw["constraints_logic"]), 5.0 * con_cov)
    algebra = safe_score(raw["algebra_validity"])
    generalization = safe_score(raw["generalization"])

    if by_kind["domain"]:
        decision = min(decision, 5.0 * dom_cov)

    score = (
        0.15 * decision
        + 0.20 * objective
        + 0.35 * constraints
        + 0.15 * algebra
        + 0.15 * generalization
    )

    if not bool(parsed["has_formulation"]):
        algebra = min(algebra, 1.0)
        generalization = min(generalization, 1.0)

        score = min(
            1.5,
            0.15 * decision
            + 0.20 * objective
            + 0.35 * constraints
            + 0.15 * algebra
            + 0.15 * generalization
        )

    all_cov = np.mean(list(status.values())) if status else np.nan

    return {
        "score_fom5_v2": round(float(score), 3),
        "decision_domains_v2": float(decision),
        "objective_v2": float(objective),
        "constraints_logic_v2": float(constraints),
        "algebra_validity_v2": float(algebra),
        "generalization_v2": float(generalization),
        "requirement_coverage_v2": float(all_cov),
        "objective_coverage": float(obj_cov),
        "constraint_coverage": float(con_cov),
        "domain_coverage": float(dom_cov),
    }


In [ ]:
CALIBRATION_CASES = [
    ("opt_02", "baseline_direct_deterministic", 2.8,
     "variable incorrecta en restricciones y término de costo omitido"),
    ("opt_10", "baseline_direct_deterministic", 1.5,
     "no entrega una formulación matemática válida"),
    ("opt_14", "baseline_direct_deterministic", 1.5,
     "intenta resolver max-flow en vez de formularlo"),
    ("opt_16", "baseline_direct_deterministic", 1.8,
     "estructura multicommodity incorrecta"),
    ("diagnostic_oftalmo", "entrega1_historical_sampled", 2.5,
     "faltan apertura/linking/ponderación y hay contradicciones"),
]

calibration_outputs = {}
calibration_rows = []

for iid, variant, expected_max, reason in tqdm(
    CALIBRATION_CASES,
    desc="Calibración local"
):
    selected = old_df[
        (old_df["id"] == iid)
        & (old_df["variant"] == variant)
    ]

    if selected.empty:
        raise ValueError(f"No encontrado: {iid} / {variant}")

    rec = selected.iloc[0]
    item = GOLD_LOOKUP[iid]

    t0 = time.time()

    parsed, raw, in_tok, out_tok = judge_local(
        item,
        rec["candidate"],
    )

    elapsed = time.time() - t0
    agg = aggregate_v2(item, parsed)

    key = (iid, variant)

    evaluation = rec.to_dict()
    evaluation.update(agg)
    evaluation.update({
        "has_formulation_v2": bool(parsed["has_formulation"]),
        "fatal_errors_v2": json.dumps(parsed.get("fatal_errors", []), ensure_ascii=False),
        "requirements_v2": json.dumps(parsed["requirements"], ensure_ascii=False),
        "judge_scores_raw_v2": json.dumps(parsed["scores"], ensure_ascii=False),
        "judge_v2_raw": raw,
        "judge_v2_error": None,
        "judge_input_tokens": in_tok,
        "judge_output_tokens": out_tok,
        "judge_latency_sec": elapsed,
        "judge_model": JUDGE_MODEL,
    })

    calibration_outputs[key] = evaluation

    calibration_rows.append({
        "id": iid,
        "variant": variant,
        "score_v2": agg["score_fom5_v2"],
        "expected_max": expected_max,
        "passes": agg["score_fom5_v2"] <= expected_max,
        "coverage": agg["requirement_coverage_v2"],
        "seconds": round(elapsed, 1),
        "reason": reason,
        "fatal_errors": parsed.get("fatal_errors", []),
    })

calibration_df = pd.DataFrame(calibration_rows)

display(calibration_df[[
    "id",
    "score_v2",
    "expected_max",
    "passes",
    "coverage",
    "seconds",
    "fatal_errors",
]])

CALIBRATION_OK = bool(calibration_df["passes"].all())

print()
if CALIBRATION_OK:
    print("CALIBRACIÓN SUPERADA.")
    print("El judge queda congelado para esta corrida.")
else:
    print("CALIBRACIÓN NO SUPERADA.")
    print("NO ejecutes el benchmark completo con este judge.")

calibration_df.to_csv(
    OUT_DIR / "calibracion_fom5_local.csv",
    index=False,
    encoding="utf-8-sig",
)


In [ ]:
assert CALIBRATION_OK, 'Calibración fallida: detener, conservar evidencia y diagnosticar sin cambiar rúbrica.'
destination=OUT_DIR/'qlora_fom5_v2_checkpoint.jsonl'
records=[json.loads(s) for s in destination.read_text(encoding='utf-8').splitlines()] if destination.exists() else []
assert len(records)==len({r['id'] for r in records})
done={r['id'] for r in records}
assert done<=set(new_df['id'])
for rec in tqdm(new_df.to_dict('records')):
    if rec['id'] in done: continue
    started=time.perf_counter()
    # Un fallo aborta sin puntuar como cero ni reducir el denominador.
    parsed,raw,in_tokens,out_tokens=judge_local(GOLD_LOOKUP[rec['id']],rec['candidate'])
    rec.update(aggregate_v2(GOLD_LOOKUP[rec['id']],parsed))
    rec.update(judge_v2_raw=raw,judge_v2_error=None,judge_model=JUDGE_MODEL,
               judge_input_tokens=in_tokens,judge_output_tokens=out_tokens,
               judge_latency_sec=time.perf_counter()-started,
               fatal_errors_v2=json.dumps(parsed['fatal_errors'],ensure_ascii=False))
    with destination.open('a',encoding='utf-8') as f: f.write(json.dumps(rec,ensure_ascii=False)+'\n')
    records.append(rec);done.add(rec['id'])
assert len(records)==31
final_df=pd.DataFrame(records)
final_df.to_csv(OUT_DIR/'qlora_fom5_v2_final.csv',index=False,encoding='utf-8-sig')
comparison=old_df[(old_df['split']=='benchmark') & (old_df['variant']=='baseline_direct_deterministic')][['id','score_fom5_v2']].merge(
    final_df[final_df['split']=='benchmark'][['id','score_fom5_v2']],on='id',validate='one_to_one',suffixes=('_base','_qlora'))
assert len(comparison)==30
comparison['delta']=comparison['score_fom5_v2_qlora']-comparison['score_fom5_v2_base']
comparison.to_csv(OUT_DIR/'paired_comparison_30.csv',index=False,encoding='utf-8-sig')
print('Principal n=30:')
display(comparison)
print(comparison.mean(numeric_only=True))
print('Caso Entrega 1:')
display(final_df[final_df['id']=='diagnostic_oftalmo'][['id','score_fom5_v2','fatal_errors_v2']])
print('Extendido n=31:',final_df['score_fom5_v2'].mean())
